# 03 — Arquitectura del Transformer (D10Sformer)

**Proyecto:** D10Sformer — MIA305 (UdeSA, 2026)  
**Fase:** 3 — Arquitectura (embeddings + atención + encoder + cabezales)

## Objetivo

1. Cargar el vocabulario construido en Fase 2 (`vocab.json`).
2. Instanciar el modelo `D10Sformer` con la configuración de `configs/base_config.yaml`.
3. Hacer un *forward pass* con un partido tokenizado real (Argentina vs. Francia hipotético) y verificar shapes.
4. Confirmar el conteo de parámetros y desglose por módulo.
5. Correr la suite de tests unitarios del Transformer.

**Nota:** este notebook NO entrena nada — solo valida que la arquitectura compila y produce shapes correctos. El pre-training real es Fase 4.

## Referencias
- Vaswani et al. (2017) *Attention Is All You Need* — atención escalada.
- Devlin et al. (2018) *BERT* — embeddings compuestos, MLM, weight tying.
- Xiong et al. (2020) *On Layer Normalization in the Transformer Architecture* — Pre-LN.

---
## 1. Setup

In [ ]:
import sys
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    PROJECT_ROOT = Path('/content/drive/MyDrive/d10sformer-v2')
    DATA_ROOT = Path('/content/drive/MyDrive/d10sformer')
else:
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
    DATA_ROOT = PROJECT_ROOT

sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from paths import ensure_paths, print_paths

paths = ensure_paths(project_root=PROJECT_ROOT, data_root=DATA_ROOT)
print_paths(paths)

# Alias legacy usados en notebooks v1
ROOT = paths.project_root
DATA_PROCESSED = paths.data_processed
CORPUS_DIR = paths.corpus_dir
VOCAB_PATH = paths.vocab_path
CKPT_DIR = paths.checkpoints
CHECKPOINTS_V1 = paths.checkpoints_v1
DATA_RAW = paths.data_raw
DATA_INTERIM = paths.data_interim


In [ ]:
import sys
from pathlib import Path

# paths: ROOT ya definido en setup
# paths: sys.path ya configurado

DATA_PROCESSED = paths.data_processed
VOCAB_PATH = paths.vocab_path

assert VOCAB_PATH.exists(), f'No encuentro {VOCAB_PATH}. Corré primero 02_tokenization.ipynb'
print(f'✓ vocab.json encontrado: {VOCAB_PATH}')

In [ ]:
import torch
import torch.nn.functional as F

from data.vocabulary import FootballVocab
from data.tokenizer import MatchTokenizer, MatchDocument, PlayerRef, MatchEvent, RollingFeatures
from models.d10sformer import D10Sformer, D10SformerConfig

print('torch:', torch.__version__)
print('CUDA disponible:', torch.cuda.is_available())
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

---
## 2. Cargar vocab + tokenizer

In [ ]:
vocab = FootballVocab.load(VOCAB_PATH)
tokenizer = MatchTokenizer(vocab, max_seq_length=512)
print(f'Vocab size: {len(vocab):,}')
print(f'PAD id: {vocab.encode("[PAD]")}')
print(f'CLS id: {vocab.encode("[CLS]")}')
print(f'MASK id: {vocab.encode("[MASK]")}')

---
## 3. Instanciar el modelo

Usamos la config por defecto: `d_model=256, num_layers=6, num_heads=8, d_ff=1024, max_seq_length=512`.

In [ ]:
config = D10SformerConfig(
    vocab_size=len(vocab),
    d_model=256,
    num_layers=6,
    num_heads=8,
    d_ff=1024,
    max_seq_length=512,
    num_segments=8,
    dropout=0.1,
    attention_dropout=0.1,
    pad_token_id=vocab.encode('[PAD]'),
    tie_mlm_weights=True,
)

model = D10Sformer(config).to(device)
print(model)

In [ ]:
breakdown = model.parameter_breakdown()
print('--- Desglose de parámetros ---')
for k, v in breakdown.items():
    pct = 100 * v / breakdown['TOTAL'] if k != 'TOTAL' else 100
    print(f'  {k:<15} {v:>12,}  ({pct:5.1f}%)')
print()
print(f'Total trainable params: {model.num_parameters():,}')
print(f'Memoria estimada (fp32): {4 * model.num_parameters() / 1e6:.1f} MB')

---
## 4. Forward pass de un partido real

Tokenizamos un partido hipotético y lo pasamos por el modelo.

In [ ]:
match = MatchDocument(
    tournament='FIFA World Cup', stage='final',
    team_a='Argentina', team_b='France', venue='neutral',
    features=RollingFeatures(home_elo=2150, away_elo=2100,
                              home_form_pts=2.5, away_form_pts=2.0,
                              home_recent_goals=2.5, away_recent_goals=1.7),
    lineup_a=[PlayerRef(player_id='5503', position='FW')] * 11,    # Messi-class
    bench_a=None,
    lineup_b=[PlayerRef(player_id='3009', position='FW')] * 11,    # Mbappé-class
    bench_b=None,
    events=[MatchEvent(minute=23, team='a', event_type='goal', player_id='5503'),
            MatchEvent(minute=67, team='b', event_type='yellow_card', player_id='3009')],
    result='home_win', home_score=2, away_score=1,
)
out = tokenizer.tokenize(match)
print(f'Tokens: {len(out.token_ids)} (max: {tokenizer.max_seq_length})')
print(f'Truncado: {out.truncated}')
print(f'Target result id: {out.target_result_id} -> {vocab.decode(out.target_result_id)}')
print(f'Target score id : {out.target_score_id} -> {vocab.decode(out.target_score_id)}')

In [ ]:
# Batch de 1
token_ids = torch.tensor(out.token_ids, dtype=torch.long, device=device).unsqueeze(0)
segment_ids = torch.tensor(out.segment_ids, dtype=torch.long, device=device).unsqueeze(0)
attention_mask = torch.ones_like(token_ids)

model.eval()
with torch.no_grad():
    logits = model(token_ids, segment_ids, attention_mask=attention_mask)

print('--- Shapes ---')
for k, v in logits.items():
    print(f'  {k:<15} {tuple(v.shape)}')

print()
print('--- Probabilidades por cabezal (modelo SIN entrenar — deben ser cercanas a uniformes) ---')
result_probs = F.softmax(logits['result_logits'], dim=-1).cpu().numpy()[0]
print(f'  P(home_win) = {result_probs[0]:.3f}')
print(f'  P(draw)     = {result_probs[1]:.3f}')
print(f'  P(away_win) = {result_probs[2]:.3f}')
print()
print(f'  Σ MLM probs (último token, sanity) = {F.softmax(logits["mlm_logits"][0, -1], dim=-1).sum().item():.4f}  (debe ser 1.0)')

---
## 5. Padding mask: batch con secuencias de distinta longitud

In [ ]:
# Partido B sparse (más corto)
sparse = MatchDocument(
    tournament='Friendly', team_a='Argentina', team_b='France', venue='neutral',
)
out_sparse = tokenizer.tokenize(sparse)
print(f'Sparse tokens: {len(out_sparse.token_ids)}')
print(f'Rich tokens:   {len(out.token_ids)}')

# Padding al máximo del batch
max_len = max(len(out.token_ids), len(out_sparse.token_ids))
pad_id = vocab.encode('[PAD]')

def pad(seq, target_len, pad_value):
    return seq + [pad_value] * (target_len - len(seq))

batch_tokens = torch.tensor([
    pad(out.token_ids, max_len, pad_id),
    pad(out_sparse.token_ids, max_len, pad_id),
], dtype=torch.long, device=device)

batch_segments = torch.tensor([
    pad(out.segment_ids, max_len, 0),
    pad(out_sparse.segment_ids, max_len, 0),
], dtype=torch.long, device=device)

batch_mask = torch.tensor([
    [1] * len(out.token_ids) + [0] * (max_len - len(out.token_ids)),
    [1] * len(out_sparse.token_ids) + [0] * (max_len - len(out_sparse.token_ids)),
], dtype=torch.long, device=device)

print(f'\nBatch shape: {batch_tokens.shape}')
print(f'Mask shape:  {batch_mask.shape}')

with torch.no_grad():
    batch_out = model(batch_tokens, batch_segments, attention_mask=batch_mask)

print(f'\nresult_logits batch shape: {batch_out["result_logits"].shape}')
print(f'Hay NaNs? {torch.isnan(batch_out["result_logits"]).any().item()}')

---
## 6. Inspección de pesos de atención

Antes del entrenamiento, las atenciones deben verse aleatorias/uniformes. Esta visualización servirá para Fase 5 (interpretabilidad — *"qué mira el modelo cuando predice un resultado"*).

In [ ]:
import matplotlib.pyplot as plt

# Forward pass por la primera capa, devolviendo pesos
model.eval()
with torch.no_grad():
    emb = model.embeddings(token_ids, segment_ids)
    # Pre-LN antes de la atención
    h = model.encoder.layers[0].ln1(emb)
    _, attn_weights = model.encoder.layers[0].attn(h, attention_mask=attention_mask, return_weights=True)

print(f'attention weights shape: {attn_weights.shape}  (B, heads, T, T)')

# Visualizar los 8 heads de la primera capa para el primer ejemplo
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for h_idx in range(8):
    ax = axes[h_idx // 4, h_idx % 4]
    ax.imshow(attn_weights[0, h_idx].cpu().numpy(), cmap='viridis', aspect='auto')
    ax.set_title(f'Layer 0, Head {h_idx}')
    ax.set_xlabel('Key position')
    ax.set_ylabel('Query position')
plt.suptitle('Atenciones de la capa 0 (modelo SIN entrenar — deben ser difusas)')
plt.tight_layout()
plt.show()

---
## 7. Backward pass de prueba (gradientes vivos)

In [ ]:
model.train()
logits = model(token_ids, segment_ids, attention_mask=attention_mask)

result_target = torch.tensor([out.target_result_id % 3], device=device)
score_target = torch.tensor([out.target_score_id], device=device)

# Vocab de score_logits es 36, así que mapeamos el target_score_id a [0..35]
# Esto es solo para validar shapes; el mapeo real (id_global → [0..35]) lo armaremos en Fase 4.
score_target_local = torch.tensor([min(out.target_score_id % 36, 35)], device=device)

loss = (
    F.cross_entropy(logits['result_logits'], result_target)
    + F.cross_entropy(logits['score_logits'], score_target_local)
)
loss.backward()
print(f'loss: {loss.item():.4f}')

# Sanity: gradiente promedio no es cero
grad_norms = []
for n, p in model.named_parameters():
    if p.grad is not None:
        grad_norms.append((n, p.grad.norm().item()))
import statistics
norms = [g for _, g in grad_norms]
print(f'\nGradientes:')
print(f'  # parámetros con grad: {len(grad_norms)}')
print(f'  ||grad|| min/median/max: {min(norms):.6f} / {statistics.median(norms):.6f} / {max(norms):.6f}')

---
## 8. Correr la suite de tests

In [ ]:
import subprocess
result = subprocess.run(
    ['python', '-m', 'pytest', str(ROOT / 'tests'), '-v', '--tb=short'],
    capture_output=True, text=True,
)
print('STDOUT:')
print(result.stdout[-3500:])
if result.stderr:
    print('\nSTDERR:')
    print(result.stderr[-1000:])
print(f'\nExit code: {result.returncode}')

---
## 9. Persistir un *dummy checkpoint* para Fase 4

Guardamos el modelo inicializado al azar para verificar que el flujo de checkpointing funciona. Fase 4 lo sobreescribirá con pesos entrenados.

In [ ]:
CKPT_DIR = paths.checkpoints
CKPT_DIR.mkdir(exist_ok=True)
CKPT_PATH = CKPT_DIR / 'd10sformer_init.pt'

torch.save({
    'model_state_dict': model.state_dict(),
    'config': config.__dict__,
}, CKPT_PATH)
size_mb = CKPT_PATH.stat().st_size / 1024 / 1024
print(f'✓ Checkpoint guardado: {CKPT_PATH} ({size_mb:.1f} MB)')

# Sanity reload
ckpt = torch.load(CKPT_PATH, map_location='cpu', weights_only=False)
model2 = D10Sformer(D10SformerConfig(**ckpt['config']))
model2.load_state_dict(ckpt['model_state_dict'])
print('✓ Reload OK')

---
## 10. Conclusiones de Fase 3

Llenar al final de la ejecución:

- [ ] Tamaño total del modelo (params): _____
- [ ] Tamaño del checkpoint (MB): _____
- [ ] Tests pasados / total: _____ / _____
- [ ] CUDA disponible: _____
- [ ] Forward pass de un partido `RICH` completa sin errores: _____
- [ ] Padding mask preserva valores finitos: _____
- [ ] Backward pass produce gradientes en todos los parámetros: _____

**Next:** Fase 4 — construir el dataset de pre-training (selección de partidos, masking estocástico de features), entrenar en clubes+selecciones (3 épocas), fine-tunear en selecciones (5 épocas), reportar perplexity de MLM y compararlo contra el baseline de Fase 1.